# 123 — Agent Cost Tracking
## What you'll learn: per-node token accounting and budget enforcement in LangGraph

⏱ ~50 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/123-agent-cost-tracking/agent_cost_tracking_workbook.ipynb)

Production agentic systems burn real money. A 3-node LangGraph agent with gpt-4o can cost $0.10–$0.50 per run. At 10,000 runs/day that's $1,000–$5,000 daily. This workbook teaches you to measure, attribute, and govern that cost.

**What you'll build:**
- `count_tokens()` — exact token counts using tiktoken (before API calls)
- `CostTracker` — per-node cost accumulator
- A 3-node LangGraph agent (planner → executor → summarizer) with cost instrumented at every node
- Budget enforcement: route to a `budget_exceeded` node if cumulative cost exceeds a limit
- A structured cost report for production cost governance

## Setup

Install dependencies and configure your API key.

In [ ]:
!pip install -q langchain-openai langgraph tiktoken python-dotenv

In [ ]:
import os

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "sk-..."  # replace with your key

## Part 1 — Why Cost Tracking Matters in Production

### The cost attribution problem

When your agent fails a budget review, which node is responsible? Without per-node tracking, you only know the total. With per-node tracking, you know:
- The executor node used 3× more tokens than the planner
- The summarizer is cheap — keep it
- The executor prompt is verbose — tighten it

### Why not just use the API response?

The OpenAI API returns token counts in `response.usage` — but only **after** the call completes. If you want to enforce a budget **before** sending a request (e.g., refuse to call the LLM if the input is too large), you need to count tokens client-side first.

tiktoken solves this: it's the same tokenizer OpenAI uses, available locally, sub-millisecond per call.

### Cost governance patterns

| Pattern | When to use |
|---------|-------------|
| Per-run budget cap | Hard limit on total agent cost |
| Per-node budget cap | Identify runaway nodes |
| Input pre-check | Refuse oversized prompts before calling API |
| Cost-based routing | Use cheap model if input < N tokens |

## Part 2 — How tiktoken Works

### BPE encoding

OpenAI models use **Byte Pair Encoding (BPE)** — a subword tokenization algorithm that merges common character pairs into single tokens. The vocabulary is learned from training data.

Key fact: **tokens ≠ words**. A word like `tokenization` is one word but may be 2–4 tokens. Short common words (`the`, `is`) are usually 1 token.

### Encoder families

| Encoder | Models |
|---------|--------|
| `cl100k_base` | gpt-4, gpt-4o, gpt-3.5-turbo, text-embedding-ada-002 |
| `o200k_base` | gpt-4o (latest), o1 |
| `p50k_base` | text-davinci-003, Codex |

`tiktoken.encoding_for_model(model_name)` selects the right encoder automatically.

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    """Count tokens in text using tiktoken."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

# Demo: count tokens for different texts
examples = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "tokenization",
    "You are a helpful assistant. Break the task into 3 numbered steps.",
    "RAG vs fine-tuning: RAG retrieves context at inference time, fine-tuning bakes knowledge into weights.",
]

print(f"{'Text':<65} {'Tokens':>6}")
print("-" * 73)
for text in examples:
    n = count_tokens(text)
    preview = text[:60] + "..." if len(text) > 60 else text
    print(f"{preview:<65} {n:>6}")

## Part 3 — OpenAI Pricing Model

### Input vs output tokens

OpenAI charges separately for input and output tokens, and output tokens are typically 3–6× more expensive:

| Model | Input (per 1K) | Output (per 1K) | Output/Input ratio |
|-------|---------------|-----------------|--------------------|
| gpt-4o | $0.0050 | $0.0150 | 3× |
| gpt-4o-mini | $0.000150 | $0.000600 | 4× |
| gpt-4-turbo | $0.0100 | $0.0300 | 3× |
| gpt-3.5-turbo | $0.0005 | $0.0015 | 3× |

**Implication:** An agent node that generates long outputs (like the executor) costs far more than one generating short outputs (like a classifier). This makes per-node tracking essential — aggregate cost hides the breakdown.

### Cost calculation

```
cost = (input_tokens / 1000) × input_price + (output_tokens / 1000) × output_price
```

In [ ]:
# OpenAI pricing table (USD per 1K tokens)
PRICING = {
    "gpt-4o":        {"input": 0.005,    "output": 0.015},
    "gpt-4o-mini":   {"input": 0.000150, "output": 0.000600},
    "gpt-4-turbo":   {"input": 0.010,    "output": 0.030},
    "gpt-3.5-turbo": {"input": 0.0005,   "output": 0.0015},
}

def estimate_cost(input_tokens: int, output_tokens: int, model: str = "gpt-4o-mini") -> float:
    """Estimate cost in USD for a single LLM call."""
    prices = PRICING.get(model, PRICING["gpt-4o-mini"])
    cost = (input_tokens / 1000) * prices["input"] + (output_tokens / 1000) * prices["output"]
    return round(cost, 8)

# Cost scenarios
scenarios = [
    ("Short QA (gpt-4o-mini)",     "gpt-4o-mini", 100, 50),
    ("Medium task (gpt-4o-mini)",   "gpt-4o-mini", 500, 300),
    ("Long analysis (gpt-4o-mini)", "gpt-4o-mini", 2000, 1000),
    ("Short QA (gpt-4o)",          "gpt-4o",      100, 50),
    ("Long analysis (gpt-4o)",     "gpt-4o",      2000, 1000),
]

print(f"{'Scenario':<35} {'In':>6} {'Out':>6} {'Cost USD':>12} {'Daily @10K':>12}")
print("-" * 75)
for label, model, inp, out in scenarios:
    cost = estimate_cost(inp, out, model)
    daily = cost * 10000
    print(f"{label:<35} {inp:>6} {out:>6} ${cost:>11.6f} ${daily:>11.2f}")

## Part 4 — CostTracker Class Design

We need an object that:
1. Travels through the LangGraph state (passed as a field in `AgentState`)
2. Accumulates a record per node: `(node_name, input_tokens, output_tokens, cost_usd)`
3. Exposes a `report()` method for post-run analysis

Using a Python `dataclass` keeps it lightweight and serialization-friendly. The `_records` list grows as the graph executes — each node appends one entry.

**Design decision:** We pass the tracker through LangGraph state (not as a global) so multiple concurrent runs don't share state. Each invocation gets its own `CostTracker()` instance.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class NodeRecord:
    node_name: str
    input_tokens: int
    output_tokens: int
    cost_usd: float

@dataclass
class CostTracker:
    """Accumulates per-node token usage and cost across a LangGraph run."""
    model: str = "gpt-4o-mini"
    _records: list = field(default_factory=list)

    def record(self, node_name: str, input_tokens: int, output_tokens: int) -> float:
        """Record a node's token usage and return the cost for this node."""
        cost = estimate_cost(input_tokens, output_tokens, self.model)
        self._records.append(NodeRecord(node_name, input_tokens, output_tokens, cost))
        return cost

    def total_cost(self) -> float:
        return round(sum(r.cost_usd for r in self._records), 8)

    def report(self) -> dict:
        node_costs = {r.node_name: r.cost_usd for r in self._records}
        most_expensive = max(self._records, key=lambda r: r.cost_usd, default=None)
        return {
            "total_cost_usd": self.total_cost(),
            "node_count": len(self._records),
            "per_node": node_costs,
            "most_expensive_node": most_expensive.node_name if most_expensive else None,
            "total_input_tokens": sum(r.input_tokens for r in self._records),
            "total_output_tokens": sum(r.output_tokens for r in self._records),
        }

# Demo: simulate 3 nodes
tracker = CostTracker(model="gpt-4o-mini")
tracker.record("planner",    120, 80)
tracker.record("executor",   350, 420)
tracker.record("summarizer", 200, 60)

import json
print(json.dumps(tracker.report(), indent=2))

## Part 5 — Instrumenting a LangGraph Node

The key pattern is a `_call_llm()` wrapper that:
1. Counts input tokens BEFORE the API call (tiktoken)
2. Calls the LLM
3. Counts output tokens AFTER the response
4. Records both to the tracker

This way we get **exact** token counts (not estimates) without relying on `response.usage` — which can differ slightly from tiktoken due to special tokens added by the API.

```python
def _call_llm(system, user, tracker, node_name):
    input_tokens = count_tokens(system + user)    # BEFORE
    response = llm.invoke([...])                   # API CALL
    output_tokens = count_tokens(response.content) # AFTER
    tracker.record(node_name, input_tokens, output_tokens)
    return response.content
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

MODEL = "gpt-4o-mini"

def _call_llm(system: str, user: str, tracker: CostTracker, node_name: str) -> str:
    """Call LLM, count tokens via tiktoken, record cost."""
    llm = ChatOpenAI(model=MODEL, temperature=0)
    input_text = system + user
    input_tokens = count_tokens(input_text, MODEL)
    response = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    output = response.content
    output_tokens = count_tokens(output, MODEL)
    cost = tracker.record(node_name, input_tokens, output_tokens)
    print(f"  [{node_name}] {input_tokens} in / {output_tokens} out / ${cost:.6f}")
    return output

print("_call_llm defined. It wraps ChatOpenAI and instruments every call with cost tracking.")

## Part 6 — The 3-Node Agent Graph

```
START → planner → executor → summarizer → END
                    ↑
            budget_exceeded → END
```

**planner:** Breaks the task into 3 numbered steps (short output — cheap).

**executor:** Executes the plan and provides results (long output — most expensive).

**summarizer:** Writes a 2-sentence summary (short output — cheap).

The `budget_exceeded` node is a dead-end — it fires if the tracker detects runaway cost.

**State design:** `CostTracker` travels in `AgentState` so each node can record its own cost without a global variable.

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):
    task: str
    plan: str
    result: str
    summary: str
    tracker: CostTracker
    budget_exceeded: bool

def planner_node(state: AgentState) -> AgentState:
    plan = _call_llm(
        "You are a task planner. Break the given task into 3 numbered steps.",
        f"Task: {state['task']}",
        state["tracker"],
        "planner",
    )
    return {**state, "plan": plan}

def executor_node(state: AgentState) -> AgentState:
    result = _call_llm(
        "You are a task executor. Execute the given plan and provide results.",
        f"Plan:\n{state['plan']}\n\nTask: {state['task']}",
        state["tracker"],
        "executor",
    )
    return {**state, "result": result}

def summarizer_node(state: AgentState) -> AgentState:
    summary = _call_llm(
        "You are a summarizer. Write a 2-sentence summary of the execution result.",
        f"Result:\n{state['result']}",
        state["tracker"],
        "summarizer",
    )
    return {**state, "summary": summary}

def budget_exceeded_node(state: AgentState) -> AgentState:
    print("  [BUDGET] Budget exceeded — stopping agent.")
    return {**state, "summary": "Stopped: budget limit reached."}

print("Nodes defined.")

In [ ]:
from langgraph.graph import StateGraph, END

tracker = CostTracker(model=MODEL)
task = "Research and explain the key differences between RAG and fine-tuning for production LLM applications."

initial_state: AgentState = {
    "task": task,
    "plan": "",
    "result": "",
    "summary": "",
    "tracker": tracker,
    "budget_exceeded": False,
}

graph = StateGraph(AgentState)
graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)
graph.add_node("summarizer", summarizer_node)
graph.add_node("budget_exceeded", budget_exceeded_node)

graph.set_entry_point("planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "summarizer")
graph.add_edge("summarizer", END)
graph.add_edge("budget_exceeded", END)

app = graph.compile()

print(f"Running agent on task: {task[:60]}...\n")
final_state = app.invoke(initial_state)

print(f"\nSummary: {final_state['summary'][:200]}")

## Part 7 — Budget Enforcement

Budget enforcement uses a **conditional edge** — LangGraph's mechanism for routing based on state.

```python
def budget_check(state) -> str:
    return "budget_exceeded" if state["tracker"].total_cost() > MAX_BUDGET else "next_node"

graph.add_conditional_edges("planner", budget_check, {
    "executor": "executor",
    "budget_exceeded": "budget_exceeded"
})
```

The check runs **after** the planner and **before** the executor — the most expensive node. If the planner already consumed too many tokens (e.g., very long system prompt), we bail early.

**Key insight:** tiktoken lets you check BEFORE making the API call. You can compute `count_tokens(system + user)` and refuse to call the API if it would exceed your budget — without spending a single token on the API.

In [ ]:
MAX_BUDGET = 0.05  # $0.05 per run

def budget_check(state: AgentState) -> str:
    """Route to budget_exceeded if cumulative cost is already too high."""
    return "budget_exceeded" if state["tracker"].total_cost() > MAX_BUDGET else "executor"

# Rebuild graph with conditional edge
graph2 = StateGraph(AgentState)
graph2.add_node("planner", planner_node)
graph2.add_node("executor", executor_node)
graph2.add_node("summarizer", summarizer_node)
graph2.add_node("budget_exceeded", budget_exceeded_node)

graph2.set_entry_point("planner")
graph2.add_conditional_edges("planner", budget_check, {
    "executor": "executor",
    "budget_exceeded": "budget_exceeded",
})
graph2.add_edge("executor", "summarizer")
graph2.add_edge("summarizer", END)
graph2.add_edge("budget_exceeded", END)

app2 = graph2.compile()
print("Graph with conditional budget check compiled.")

In [ ]:
# Demo: set budget to $0.000001 to trigger the budget exceeded path
MAX_BUDGET = 0.000001

tracker2 = CostTracker(model=MODEL)
initial_state2: AgentState = {
    "task": task,
    "plan": "",
    "result": "",
    "summary": "",
    "tracker": tracker2,
    "budget_exceeded": False,
}

print(f"Budget limit: ${MAX_BUDGET:.6f} (intentionally tiny to trigger budget exceeded)\n")
final_state2 = app2.invoke(initial_state2)
print(f"\nOutcome: {final_state2['summary']}")
print(f"Cost after planner: ${tracker2.total_cost():.6f}")

## Part 8 — Cost Report Interpretation

The `tracker.report()` method returns a dict with:

| Field | Meaning |
|-------|---------|
| `total_cost_usd` | Total cost for this run |
| `node_count` | Number of nodes that executed |
| `per_node` | `{node_name: cost_usd}` for every node |
| `most_expensive_node` | Name of the node with highest cost |
| `total_input_tokens` | Sum of input tokens across all nodes |
| `total_output_tokens` | Sum of output tokens across all nodes |

**What to do with this in production:**
- Log to a database (Postgres, BigQuery) for trend analysis
- Alert if a node exceeds its per-node budget
- Use `most_expensive_node` to prioritize prompt optimization
- Aggregate across users to compute cost per feature

In [ ]:
import json

# Cost report from the full run (Cell 15)
report = tracker.report()

print("=== Cost Report ===")
print(json.dumps(report, indent=2))

print(f"\n--- Summary ---")
print(f"Total cost:           ${report['total_cost_usd']:.6f}")
print(f"Most expensive node:  {report['most_expensive_node']}")
print(f"Total tokens used:    {report['total_input_tokens'] + report['total_output_tokens']}")

# Cost breakdown as percentage
if report['total_cost_usd'] > 0:
    print("\n--- Per-node breakdown ---")
    for node, cost in report['per_node'].items():
        pct = (cost / report['total_cost_usd']) * 100
        print(f"  {node:<15} ${cost:.6f}  ({pct:.1f}%)")

## Exercises

**Exercise 1:** Add a 4th `validator` node after the summarizer. The validator calls the LLM to check if the summary is accurate. Track its cost separately. Does it dominate the cost profile?

**Exercise 2:** Using the cost report from your run, calculate:
- How many runs fit within a $1.00 daily budget?
- What is the most expensive operation as a % of total cost?
- If you switched from gpt-4o-mini to gpt-4o, how much would the same run cost?

In [ ]:
# Exercise 1: Add a validator node

def validator_node(state: AgentState) -> AgentState:
    # TODO: call the LLM to verify the summary is accurate
    # Hint: use _call_llm() with a validation prompt
    # Record cost under node_name="validator"
    pass

# TODO: rebuild the graph with validator after summarizer
# graph3 = StateGraph(AgentState)
# ... add nodes ...
# graph3.add_edge("summarizer", "validator")
# graph3.add_edge("validator", END)

In [ ]:
# Exercise 2: Budget math

# Use the report from your run above
cost_per_run = report['total_cost_usd']

daily_budget = 1.00

# How many runs fit in the daily budget?
# TODO: calculate and print

# What is the executor's share of total cost?
# TODO: calculate executor % of total

# What would the same run cost on gpt-4o?
# Hint: re-run estimate_cost() with gpt-4o pricing and the same token counts
# TODO: calculate gpt-4o cost

## Answer Key

In [ ]:
# Answer key

# Exercise 1: validator node
def validator_node_answer(state: AgentState) -> AgentState:
    validation = _call_llm(
        "You are a fact-checker. Verify the summary is accurate and complete. Reply YES or NO with a brief reason.",
        f"Summary: {state['summary']}\n\nOriginal result: {state['result'][:500]}",
        state["tracker"],
        "validator",
    )
    return {**state, "summary": state["summary"] + f"\n[Validated: {validation[:80]}]"}

# Exercise 2: budget math
if cost_per_run > 0:
    runs_per_day = int(daily_budget / cost_per_run)
    print(f"Runs per day at ${daily_budget:.2f}: {runs_per_day:,}")

    executor_cost = report['per_node'].get('executor', 0)
    executor_pct = (executor_cost / cost_per_run) * 100
    print(f"Executor % of total: {executor_pct:.1f}%")

    # gpt-4o equivalent cost
    gpt4o_cost = estimate_cost(
        report['total_input_tokens'],
        report['total_output_tokens'],
        "gpt-4o"
    )
    multiplier = gpt4o_cost / cost_per_run if cost_per_run > 0 else 0
    print(f"Same run on gpt-4o: ${gpt4o_cost:.6f} ({multiplier:.1f}x more expensive)")

## Workshop Complete

You have built a production-grade cost tracking layer for LangGraph agents:

- **tiktoken** for client-side token counting (before the API call)
- **CostTracker** dataclass for per-node cost accumulation
- **3-node LangGraph agent** with cost instrumented at every step
- **Conditional budget enforcement** using LangGraph's routing mechanism
- **Structured cost reports** for production observability

### Next steps

- **Example 124:** Crescendo multi-turn attack — cost tracking in adversarial settings
- **Log to Postgres:** Insert `report` into a `agent_runs` table for trend dashboards
- **Alerting:** Send a Slack/email alert when `total_cost_usd > THRESHOLD`
- **Model routing:** Use `count_tokens()` pre-call to route short tasks to gpt-4o-mini and complex tasks to gpt-4o